# HDD 09 — Q3 development expansion and acceptance reservation

Run in VS Code after Notebook 08. This creates a **new development dataset**, preserving the original experiments and June final-test reservation. It does not train or score a model.

Download **2021 Q3** from [Backblaze's official data page](https://www.backblaze.com/cloud-storage/resources/hard-drive-test-data) and put the ZIP at `data/raw/data_Q3_2021.zip`. Keep it zipped: daily members are streamed in chunks. Alternatively change `Q3_ZIP` in the setup cell to your downloaded file. The provider lists this archive at approximately 974 MB. Do not use Q2 here.

Protocol: select drives present on July 1 using serial/model identifiers only; exclude every original reserved serial; deterministically reserve up to 200 adaptation and 100 acceptance drives. Group assignment never uses survival, failure labels, SMART values or future completeness. Adaptation endpoints: July 30–August 31. Acceptance endpoints: September 1–30 on separate drives, with preceding history used only as window context. The acceptance set is a development proxy, not certified healthy data or a new final test.

This single-model expansion does not satisfy the proposal's eventual multi-model and multiple-seed requirements.

In [ ]:
from pathlib import Path
import json,hashlib,re,zipfile
from datetime import datetime,timezone
import numpy as np
import pandas as pd
from IPython.display import display
COMPONENT_ROOT=Path(r'D:/SLIIT/Y4S1/RP/Research/J26-DS-342-/01_component_1_behavioral_drift_genai')
RUN_ID='single_model_q1q2_2021_v1'
EXPANSION_ID='q3_development_expansion_v1'
MODEL='ST12000NM0008'
Q3_ZIP=COMPONENT_ROOT/'data/raw/data_Q3_2021.zip'
OUT=COMPONENT_ROOT/'results'/RUN_ID/EXPANSION_ID
DATA=COMPONENT_ROOT/'data/features'/RUN_ID/EXPANSION_ID
for p in [OUT,DATA]:p.mkdir(parents=True,exist_ok=True)
def sha256(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()
up=COMPONENT_ROOT/'results'/RUN_ID/'quarantine_replay_readiness_v1/replay_buffer_complete.json'
assert up.is_file(),f'Complete Notebook 08 first: {up}'
record=json.loads(up.read_text())
assert record['analysis_complete'] and not record['sealed_test_opened'] and not record['real_failure_labels_used']
assert Q3_ZIP.is_file(),f'Download the 2021 Q3 ZIP to {Q3_ZIP}, or edit Q3_ZIP.'
old_path=COMPONENT_ROOT/'data/splits'/f'{RUN_ID}_serials.json'
old=json.loads(old_path.read_text())
# Existing split JSON contains named lists of serial numbers. Refuse an unknown schema.
assert all(isinstance(v,list) and all(isinstance(s,str) for s in v) for v in old.values()), 'Unexpected original split schema; inspect before proceeding.'
excluded=set().union(*(set(v) for v in old.values()))
assert {'train','calibration','development'}.issubset(old)
assert any('test' in k.lower() for k in old), 'Final-test serial reservation must be included.'
FEATURES=['smart_5_raw','smart_9_raw','smart_194_raw','smart_197_raw','smart_198_raw']
SETTINGS={'model':MODEL,'features':FEATURES,'window_days':30,'cohort_date':'2021-07-01',
    'adaptation_start':'2021-07-30','adaptation_end':'2021-08-31',
    'acceptance_start':'2021-09-01','acceptance_end':'2021-09-30',
    'adaptation_target':200,'acceptance_target':100,'minimum_drives_per_group':30,
    'assignment_salt':'q3-development-v1-seed42','missing_rule':'reject any window with absent day or nonfinite/negative feature',
    'upstream_complete_sha256':sha256(up),'original_splits_sha256':sha256(old_path),
    'archive_sha256':sha256(Q3_ZIP),'failure_labels_read':False}
p=OUT/'settings.json'
if p.exists():assert json.loads(p.read_text())==SETTINGS,'Changed protocol/input: use a new EXPANSION_ID.'
p.write_text(json.dumps(SETTINGS,indent=2))
print('Archive verified locally; hashing records identity, not provider authenticity.')
print('Previously reserved serials excluded:',len(excluded))

## 1. Freeze a July 1 cohort before reading subsequent telemetry

A missing archive day or duplicate dated member stops the run. Extra metadata files are ignored. The archive must contain all 92 Q3 calendar dates; rows from June are never opened.

In [ ]:
def dated_members(z):
    members={}
    for info in z.infolist():
        if info.is_dir() or '__MACOSX' in info.filename:continue
        match=re.fullmatch(r'(2021-\d{2}-\d{2})\.csv',Path(info.filename).name)
        if not match:continue
        date=match.group(1)
        if not ('2021-07-01'<=date<='2021-09-30'):continue
        assert date not in members,f'Duplicate dated archive member: {date}'
        members[date]=info.filename
    expected=set(pd.date_range('2021-07-01','2021-09-30').strftime('%Y-%m-%d'))
    assert set(members)==expected,f'Missing Q3 dates: {sorted(expected-set(members))}'
    return members
with zipfile.ZipFile(Q3_ZIP) as z:
    MEMBERS=dated_members(z)
    with z.open(MEMBERS['2021-07-01']) as f:
        cohort=pd.read_csv(f,usecols=['serial_number','model'],dtype='string')
assert not cohort.serial_number.isna().any()
assert not cohort.serial_number.duplicated().any(), 'Duplicate July 1 serial.'
ids=sorted(set(cohort.loc[cohort.model.eq(MODEL),'serial_number'])-excluded)
assert len(ids)>=2*SETTINGS['minimum_drives_per_group'], 'Insufficient unseen drives: report this result; do not recycle reserved test drives.'
def rank(s):return hashlib.sha256((SETTINGS['assignment_salt']+'|'+s).encode()).hexdigest()
ordered=sorted(ids,key=rank)
# If availability is lower than 300, keep a roughly 2:1 split with >=30 each.
count=min(len(ids),SETTINGS['adaptation_target']+SETTINGS['acceptance_target'])
n_accept=max(SETTINGS['minimum_drives_per_group'],int(round(count/3)))
n_adapt=count-n_accept
GROUPS={'adaptation':ordered[:n_adapt],'acceptance':ordered[n_adapt:count]}
assert min(map(len,GROUPS.values()))>=SETTINGS['minimum_drives_per_group']
assert set(GROUPS['adaptation']).isdisjoint(GROUPS['acceptance'])
assert set().union(*map(set,GROUPS.values())).isdisjoint(excluded)
split_path=OUT/'q3_serials.json'
if split_path.exists():assert json.loads(split_path.read_text())==GROUPS,'Cohort changed. Use a new EXPANSION_ID.'
split_path.write_text(json.dumps(GROUPS,indent=2))
SPLIT_HASH=sha256(split_path)
print('Frozen groups:',{k:len(v) for k,v in GROUPS.items()})
print('Assignment used identifiers on July 1 only.')

## 2. Stream selected records and audit data quality

Only identifiers, dates and the five declared features are read. The failure column is excluded at CSV parsing. No model is fit and no feature distribution is used to select drives. Values are stored in original units. Missing days and values are retained for window rejection and coverage reporting; no surviving-drive filter is applied.

In [ ]:
selected=set().union(*map(set,GROUPS.values()))
frames=[];daily_audit=[]
with zipfile.ZipFile(Q3_ZIP) as z:
    for date,member in sorted(MEMBERS.items()):
        pieces=[]
        with z.open(member) as f:
            for chunk in pd.read_csv(f,usecols=['date','serial_number','model']+FEATURES,
                    dtype={'serial_number':'string','model':'string'},chunksize=50000):
                keep=chunk.loc[chunk.serial_number.isin(selected)].copy()
                if not keep.empty:pieces.append(keep)
        day=pd.concat(pieces,ignore_index=True) if pieces else pd.DataFrame(columns=['date','serial_number','model']+FEATURES)
        if not day.empty:
            assert day.model.eq(MODEL).all(),'Selected serial changed HDD model.'
            day['date']=pd.to_datetime(day.date,errors='raise').dt.normalize()
            assert day.date.eq(pd.Timestamp(date)).all(),'Archive filename/date mismatch.'
            assert not day.duplicated(['serial_number','date']).any(),'Duplicate drive-date records.'
            for feature in FEATURES:
                day[feature]=pd.to_numeric(day[feature],errors='raise')
            frames.append(day)
        daily_audit.append({'date':date,'selected_observed_drives':len(day),'selected_panel_size':len(selected)})
        if date.endswith(('01','15')):print('Read',date)
daily=pd.concat(frames,ignore_index=True).sort_values(['serial_number','date']).reset_index(drop=True)
assert not daily.duplicated(['serial_number','date']).any()
assert 'failure' not in daily.columns
pd.DataFrame(daily_audit).to_csv(OUT/'source_daily_coverage.csv',index=False)
for name,serials in GROUPS.items():
    folder=DATA/name;folder.mkdir(exist_ok=True)
    part=daily.loc[daily.serial_number.isin(serials)].reset_index(drop=True)
    part.to_parquet(folder/'daily_raw.parquet',index=False)
print('Selected daily rows saved; acceptance telemetry has not been scored or used to fit anything.')

## 3. Build independent 30-day windows

The tensor feature order is frozen. Each window requires 30 consecutive calendar dates and all five finite, nonnegative SMART values. Rejection logs retain reasons and original observation counts. Accepted masks are all true because this first expansion uses complete windows only. No preprocessing parameters are fit here: the following notebook must apply the original training-only preprocessing to compare the frozen model with a candidate.

Acceptance windows may use August context from **acceptance drives only**. These rows are never added to adaptation training. Keep acceptance tensors closed during candidate selection; the saved count audit checks feasibility without reporting score performance.

In [ ]:
def build_windows(part,serials,start,end):
    groups={str(s):g.set_index('date') for s,g in part.groupby('serial_number')}
    X=[];masks=[];meta=[];audit=[]
    for serial in serials:
        g=groups.get(serial,pd.DataFrame(columns=FEATURES,index=pd.DatetimeIndex([])))
        for endpoint in pd.date_range(start,end):
            dates=pd.date_range(endpoint-pd.Timedelta(days=29),endpoint)
            v=g.reindex(dates)[FEATURES].to_numpy(dtype=np.float64)
            observed=np.isfinite(v)&(v>=0)
            days_present=int(dates.isin(g.index).sum())
            reason='accepted' if observed.all() else ('missing_calendar_days' if days_present<30 else 'invalid_or_missing_feature')
            audit.append({'serial_number':serial,'end_date':endpoint,'reason':reason,
                'days_present':days_present,'observed_values':int(observed.sum())})
            if reason=='accepted':
                X.append(v);masks.append(observed)
                meta.append({'serial_number':serial,'start_date':dates[0],'end_date':endpoint})
    a=np.stack(X) if X else np.empty((0,30,len(FEATURES)),dtype=np.float64)
    m=np.stack(masks) if masks else np.empty(a.shape,dtype=bool)
    return a,m,pd.DataFrame(meta,columns=['serial_number','start_date','end_date']),pd.DataFrame(audit)

counts=[];WINDOW_META={}
for name,serials in GROUPS.items():
    folder=DATA/name
    part=pd.read_parquet(folder/'daily_raw.parquet')
    start,end=(SETTINGS['adaptation_start'],SETTINGS['adaptation_end']) if name=='adaptation' else (SETTINGS['acceptance_start'],SETTINGS['acceptance_end'])
    X,mask,meta,audit=build_windows(part,serials,start,end)
    np.savez_compressed(folder/'windows_raw.npz',X=X,observed_mask=mask,features=np.array(FEATURES))
    meta.to_parquet(folder/'metadata.parquet',index=False)
    audit.to_csv(OUT/f'{name}_window_audit.csv',index=False)
    WINDOW_META[name]=meta
    assert len(meta)==len(X)==len(mask)
    assert mask.all() and np.isfinite(X).all()
    assert not meta.duplicated(['serial_number','end_date']).any()
    counts.append({'split':name,'declared_drives':len(serials),'scoreable_drives':meta.serial_number.nunique(),
        'requested_windows':len(audit),'accepted_windows':len(meta),'rejected_windows':len(audit)-len(meta),
        'endpoint_dates_with_windows':meta.end_date.nunique()})
counts=pd.DataFrame(counts)
counts.to_csv(OUT/'expansion_window_counts.csv',index=False)
display(counts)
assert set(WINDOW_META['adaptation'].serial_number).isdisjoint(WINDOW_META['acceptance'].serial_number)
assert (counts.accepted_windows+counts.rejected_windows).eq(counts.requested_windows).all()
feasible=bool(counts.scoreable_drives.ge(SETTINGS['minimum_drives_per_group']).all() and counts.accepted_windows.gt(0).all())
print('Window preparation feasible:',feasible,'| This does not validate adaptation or health.')

## 4. Verify outputs and save provenance

The next notebook can perform a frozen-model Q3 development diagnostic on the adaptation group and implement offline candidate fitting. Before acceptance scoring, it must define reconstruction and alert-budget tolerances and a limit on repeated acceptance comparisons. Failure-label warning evaluation remains a separate later stage. Since Q3 is now development, any later temporal final evaluation must use a newly reserved future period; June remains an earlier untouched benchmark and cannot represent a future-of-Q3 test.

In [ ]:
assert sha256(old_path)==SETTINGS['original_splits_sha256']
assert sha256(up)==SETTINGS['upstream_complete_sha256']
assert sha256(split_path)==SPLIT_HASH
assert sha256(Q3_ZIP)==SETTINGS['archive_sha256'],'Archive changed during preparation.'
artifacts={str(p.relative_to(COMPONENT_ROOT)):sha256(p) for p in sorted(DATA.rglob('*')) if p.is_file()}
artifacts.update({str(p.relative_to(COMPONENT_ROOT)):sha256(p) for p in sorted(OUT.glob('*.csv'))})
completion={'created_utc':datetime.now(timezone.utc).isoformat(),'analysis_complete':True,
    'dataset_feasible':feasible,'run_id':RUN_ID,'expansion_id':EXPANSION_ID,'settings':SETTINGS,
    'split_sha256':SPLIT_HASH,'artifact_sha256':artifacts,'failure_labels_read':False,
    'sealed_June_test_opened':False,'acceptance_scores_inspected':False,'models_updated':0,
    'limitations':['one model','development acceptance is not a healthy-labelled population',
        'Q3 is development and later than the reserved June benchmark','no scaling, inference or training performed',
        'complete-window rejection can produce coverage bias; retain audit counts']}
(OUT/'expansion_complete.json').write_text(json.dumps(completion,indent=2,allow_nan=False))
print('NOTEBOOK 09 COMPLETE — Q3 development dataset and acceptance reservation saved.')
print('Dataset feasible:',feasible)
print('Share this executed notebook and expansion_window_counts.csv.')
print('Results:',OUT)

Commit message:

`feat: prepare Q3 development windows and reserve drive-separated acceptance data`

Keep ZIPs, daily data and window tensors out of ordinary Git commits. Commit code, configuration and small audit outputs according to your team policy.